<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/day1/notebooks/3_bow_approaches_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Day 2 — Bag-of-Words Approaches: Dictionary, Supervised ML & Topic Modeling

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

So far we have turned text into a document-term matrix. Now we put that matrix to **work**
using the three classic families of bag-of-words methods:

1. **Dictionary methods** — count words from a predefined list (rule-based, no learning)
2. **Supervised machine learning** — learn to predict labels from labeled examples
3. **Topic modeling** — discover themes with no labels at all (unsupervised)

We use a real, labeled social-science dataset: **TweetEval** (Barbieri et al. 2020), a
widely used academic benchmark of tweets. We work with the **sentiment** subset, where
each tweet is labeled negative, neutral, or positive.

By the end you will be able to:

- Apply a sentiment **dictionary** and evaluate it against human labels
- Train and evaluate a **supervised classifier** on text
- Fit and interpret a **topic model** (LDA)
- Understand where each approach shines and where it struggles

> Run each cell in order with `Shift + Enter`. Try each **✏️ Exercise** before moving on.


## 0. Setup

`datasets` and `wordcloud` may need installing in Colab — the line below handles that.


In [ ]:
# Install libraries that may not be pre-installed in Colab
!pip install datasets wordcloud -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.decomposition import LatentDirichletAllocation

nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("wordnet")

print("Setup complete!")

## 1. Loading the TweetEval sentiment data

TweetEval is hosted on the Hugging Face Hub and loads in one line. The sentiment subset
labels each tweet as:

- **0 = negative**
- **1 = neutral**
- **2 = positive**


In [ ]:
from datasets import load_dataset

# Load the sentiment subset of TweetEval
dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")

# Convert the train and test splits to pandas DataFrames
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

print("Train size:", len(train_df))
print("Test size: ", len(test_df))
train_df.head()

Let's look at the label distribution and a few example tweets.


In [ ]:
label_names = {0: "negative", 1: "neutral", 2: "positive"}
train_df["label_name"] = train_df["label"].map(label_names)
test_df["label_name"] = test_df["label"].map(label_names)

print("Label distribution (train):")
print(train_df["label_name"].value_counts())
print()
for lab in [0, 1, 2]:
    example = train_df[train_df["label"] == lab]["text"].iloc[0]
    print(f"[{label_names[lab]}] {example}")

> **✏️ Exercise 1**
>
> Print **three** example tweets for the *negative* class (label 0). Do the human labels
> look reasonable to you? Sentiment labeling is famously subjective — see if you agree.


In [ ]:
# Your code here


### A note on size

The full training set is large. To keep everything fast and responsive in a workshop
setting, we work with a **random sample**. For a real analysis you would use all of it.


In [ ]:
# Take a manageable sample for speed (set to a larger number for real work)
SAMPLE_SIZE = 3000

train_sample = train_df.sample(n=min(SAMPLE_SIZE, len(train_df)), random_state=42).reset_index(drop=True)
test_sample = test_df.sample(n=min(1000, len(test_df)), random_state=42).reset_index(drop=True)

print("Working sample sizes:")
print("  Train:", len(train_sample))
print("  Test: ", len(test_sample))

## 2. Light preprocessing

Tweets are messy: mentions (`@user`), hashtags, URLs, emoji. We do a light clean. Note we
keep it lighter than before — for sentiment, aggressive cleaning can remove useful signal.


In [ ]:
import re

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_tweet(text):
    """Light cleaning for tweets."""
    text = text.lower()
    text = re.sub(r"http\S+", "", text)      # remove URLs
    text = re.sub(r"@\w+", "", text)          # remove @mentions
    text = re.sub(r"#", "", text)              # keep hashtag word, drop the #
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha()]
    return " ".join(tokens)

# Test on one tweet
print("RAW:    ", train_sample["text"].iloc[0])
print("CLEANED:", clean_tweet(train_sample["text"].iloc[0]))

In [ ]:
train_sample["clean"] = train_sample["text"].apply(clean_tweet)
test_sample["clean"] = test_sample["text"].apply(clean_tweet)
print("Done cleaning.")

## 3. Approach 1 — Dictionary methods

The simplest approach: define lists of positive and negative words, count them, and score
each tweet. No training required. We'll use a small hand-built dictionary for transparency.


In [ ]:
positive_words = {
    "good", "great", "love", "happy", "excellent", "wonderful", "best",
    "amazing", "awesome", "nice", "beautiful", "fantastic", "perfect",
    "thanks", "thank", "glad", "excited", "enjoy", "win", "congratulations",
}
negative_words = {
    "bad", "hate", "terrible", "awful", "worst", "horrible", "sad", "angry",
    "disappointed", "disappointing", "poor", "ugly", "wrong", "fail", "sucks",
    "boring", "annoying", "useless", "waste", "sorry",
}

def dictionary_sentiment(text):
    """Score a tweet: +1 per positive word, -1 per negative word."""
    tokens = text.split()
    score = sum(1 for t in tokens if t in positive_words)
    score -= sum(1 for t in tokens if t in negative_words)
    if score > 0:
        return 2   # positive
    elif score < 0:
        return 0   # negative
    else:
        return 1   # neutral

# Apply to the test sample
test_sample["dict_pred"] = test_sample["clean"].apply(dictionary_sentiment)

# Show a few
test_sample[["text", "label_name", "dict_pred"]].head(8)

How well does the dictionary do? Because we have **human labels**, we can measure it.


In [ ]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(test_sample["label"], test_sample["dict_pred"])
print(f"Dictionary accuracy: {acc:.3f}")
print()
print(classification_report(
    test_sample["label"], test_sample["dict_pred"],
    target_names=["negative", "neutral", "positive"],
    zero_division=0,
))

> **✏️ Exercise 2**
>
> The dictionary is small. Add a few of your own words to `positive_words` and
> `negative_words`, re-run the scoring, and see whether accuracy improves. What are the
> limits of just adding more words?


In [ ]:
# Your code here


**Reflection.** Notice the dictionary is completely **transparent** — you can see
exactly why any tweet got its score — but it is **context-blind**. It cannot handle
*"not good"*, sarcasm, or words it does not know. This is the classic dictionary trade-off.


## 4. Approach 2 — Supervised machine learning

Instead of specifying words ourselves, we let a model **learn** which words predict which
label, from the labeled training data.

The workflow:
1. Turn text into a numerical matrix (TF-IDF this time)
2. Train a classifier on the **training** set
3. Evaluate on the **test** set (data the model never saw)


In [ ]:
# 1. Vectorize: fit on train, apply to test
vectorizer = TfidfVectorizer(max_features=5000)
X_train = vectorizer.fit_transform(train_sample["clean"])
X_test = vectorizer.transform(test_sample["clean"])

y_train = train_sample["label"]
y_test = test_sample["label"]

print("Training matrix shape:", X_train.shape)

In [ ]:
# 2. Train a logistic regression classifier
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# 3. Predict and evaluate on the test set
y_pred = clf.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print()
print(classification_report(
    y_test, y_pred,
    target_names=["negative", "neutral", "positive"],
    zero_division=0,
))

Already better than the dictionary, usually — because the model **learned** from data
rather than relying on a fixed list. Let's see *what* it learned: the most predictive words
for each class.


In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())

for class_idx, class_name in enumerate(["negative", "neutral", "positive"]):
    top = np.argsort(clf.coef_[class_idx])[-10:][::-1]
    print(f"Top words for '{class_name}':")
    print("  ", ", ".join(feature_names[top]))
    print()

> **✏️ Exercise 3**
>
> Swap the logistic regression for **Multinomial Naive Bayes** (already imported as
> `MultinomialNB`). Train it the same way and compare accuracy. Which does better here?
>
> *(Note: Naive Bayes needs non-negative features — TF-IDF is fine.)*


In [ ]:
# Your code here


### Visualizing errors: the confusion matrix

A confusion matrix shows *where* the model gets confused — which true labels get predicted
as which.


In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Greens")
labels = ["negative", "neutral", "positive"]
ax.set_xticks(range(3)); ax.set_xticklabels(labels)
ax.set_yticks(range(3)); ax.set_yticklabels(labels)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion matrix (supervised classifier)")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im)
plt.tight_layout()
plt.show()

> **✏️ Exercise 4**
>
> Look at the confusion matrix. Which class is hardest for the model? (Hint: neutral tweets
> are often the trickiest — think about why that might be.)


In [ ]:
# Your code here


## 5. Approach 3 — Topic modeling (unsupervised)

Now we **throw away the labels** entirely and ask a different question: *what themes run
through these tweets?* Topic modeling (here, **LDA**) discovers clusters of co-occurring
words with no supervision.

Topic models work on **counts**, so we use `CountVectorizer` (not TF-IDF), and remove
stopwords so topics are about content.


In [ ]:
# Vectorize with counts, dropping stopwords and very common/rare terms
count_vec = CountVectorizer(
    max_features=1000,
    stop_words="english",
    min_df=5,
)
X_counts = count_vec.fit_transform(train_sample["clean"])

print("Matrix for topic modeling:", X_counts.shape)

In [ ]:
# Fit LDA with a chosen number of topics
N_TOPICS = 5

lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=42,
    max_iter=10,
)
lda.fit(X_counts)

print(f"Fitted an LDA model with {N_TOPICS} topics.")

Each topic is a distribution over words. Let's print the **top words** per topic — and
remember: *we* have to interpret and label them.


In [ ]:
def show_topics(model, feature_names, n_words=10):
    for idx, topic in enumerate(model.components_):
        top = topic.argsort()[-n_words:][::-1]
        words = [feature_names[i] for i in top]
        print(f"Topic {idx}: {', '.join(words)}")

feature_names = count_vec.get_feature_names_out()
show_topics(lda, feature_names)

> **✏️ Exercise 5**
>
> Read the topics above and try to give each one a **human label** (e.g. "sports",
> "politics", "everyday chat"). Some will be coherent; some may be junk — that is normal
> and part of the interpretive work topic modeling requires.


In [ ]:
# Your interpretation here (as a comment or in a new markdown cell)


> **✏️ Exercise 6**
>
> Re-fit the LDA model with a **different number of topics** (try `N_TOPICS = 3` and
> `N_TOPICS = 8`). How does the interpretability change? There is no single 'right' number
> — it is a judgment call.


In [ ]:
# Your code here


## 6. Putting it together

You have now applied all three classic bag-of-words approaches to the same corpus:

| Approach | Learning | You provide | We measured |
|---|---|---|---|
| **Dictionary** | None (rule-based) | A word list | Accuracy vs. labels |
| **Supervised ML** | Supervised | Labeled examples | Accuracy, top words, errors |
| **Topic modeling** | Unsupervised | Nothing but text | Interpretable topics |

Key takeaways:

- The **dictionary** is transparent but context-blind and usually least accurate.
- **Supervised ML** learns from data and typically wins on accuracy — but needs labels.
- **Topic modeling** needs no labels and is great for exploration — but requires
  interpretation and careful choices.

These are not rivals. A common real workflow: **topic model** to explore, a **dictionary**
to measure a well-defined construct, and a **classifier** to scale up labeling.

### Optional challenge

Everything here used bag-of-words. Its core weakness (from the lecture): *"not good"* and
*"good"* look almost identical to these models. Find a tweet in the test set where the
**dictionary got it wrong because of negation or sarcasm**, and write a sentence explaining
what went wrong. This is exactly the gap that **embeddings** — our next topic — start to close.


In [ ]:
# Optional challenge — your code here
